**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 07 - Modelado de problemas organizacionales mediante sistemas de ecuaciones lineales**

## Complemento

Este notebook se complementa con la presentación: **AlgebrayORG.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

## ¿Qué vamos a hacer en esta clase?

La clase pasada aprendimos a operar con matrices. Hoy las usamos para lo que fueron inventadas:
**resolver varios problemas entrelazados al mismo tiempo**.

| Parte | Tema | Herramienta |
|---|---|---|
| **A** | De un problema de gestión a un sistema de ecuaciones | Lápiz y papel |
| **B** | Resolverlo en Python | `np.linalg.solve` |
| **C** | Cuándo el sistema **no** tiene solución | El determinante |
| **D** | Un caso más grande: planificación de producción | Sistema 3×3 |
| **E** | Un proceso P2P que se muerde la cola | Carga real y dimensionamiento |
| **F** | El costo real de procesar una factura | Costeo de servicios compartidos |

> **La idea de fondo:** cuando una decisión depende de otra, que depende de otra, no alcanza con
> despejar de a una. Hay que resolverlas todas juntas. Eso es un sistema de ecuaciones.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)

# RAMA: de qué versión del repositorio se leen los datos.
# Si algún archivo te da error 404, probá con RAMA = "juan"
RAMA = "main"
URL = f"https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/{RAMA}/DF/"

---
# 🏭 Parte A — De un problema de gestión a un sistema

**El caso.** Una fábrica produce **sillas** y **mesas**. Cada una consume horas de dos sectores:

| | Carpintería | Pintura |
|---|---|---|
| **Una silla** | 2 hs | 1 h |
| **Una mesa** | 4 hs | 3 hs |
| **Disponible por semana** | **220 hs** | **150 hs** |

**La pregunta del gerente:** ¿cuántas sillas y cuántas mesas hay que producir para usar
*exactamente* toda la capacidad instalada, sin que sobren ni falten horas?

---

**Paso 1 — Nombrar las incógnitas.**

$$s = \text{cantidad de sillas} \qquad m = \text{cantidad de mesas}$$

**Paso 2 — Escribir una ecuación por cada restricción.**

$$\begin{cases} 2s + 4m = 220 & \text{(horas de carpintería)} \\ 1s + 3m = 150 & \text{(horas de pintura)} \end{cases}$$

**Paso 3 — Reescribirlo como matrices.** Esta es la traducción clave:

$$\underbrace{\begin{pmatrix} 2 & 4 \\ 1 & 3 \end{pmatrix}}_{A \ \text{(tecnología)}} \cdot \underbrace{\begin{pmatrix} s \\ m \end{pmatrix}}_{x \ \text{(incógnitas)}} = \underbrace{\begin{pmatrix} 220 \\ 150 \end{pmatrix}}_{b \ \text{(recursos)}}$$

En forma compacta: $$A \cdot x = b$$

- **A** describe *cómo* produce la empresa (la tecnología). Cada **fila** es un recurso, cada **columna** un producto.
- **b** es *cuánto* tiene disponible.
- **x** es lo que queremos averiguar.

---
# 🐍 Parte B — Resolverlo en Python

Si esto fuera álgebra común, despejaríamos $x = b / A$. Con matrices la división no existe, pero
existe algo equivalente: multiplicar por la **matriz inversa**.

$$A \cdot x = b \quad \Longrightarrow \quad x = A^{-1} \cdot b$$

En Python no hace falta ni calcular la inversa: `np.linalg.solve` lo hace de una y es más preciso.

In [ ]:
A = np.array([[2, 4],      # fila 1: carpintería  → 2 hs por silla, 4 hs por mesa
              [1, 3]])     # fila 2: pintura      → 1 h por silla,  3 hs por mesa

b = np.array([220, 150])   # horas disponibles de cada sector

x = np.linalg.solve(A, b)  # resolver el sistema A·x = b

print("Sillas:", round(x[0], 2))
print("Mesas: ", round(x[1], 2))

**Nunca confíes en un resultado sin verificarlo.** Volvemos a multiplicar `A · x` y tiene que dar `b`:

In [ ]:
verificacion = A @ x     # el símbolo @ multiplica matrices (NO uses * , eso multiplica elemento a elemento)

print("A · x =", verificacion)
print("b     =", b)
print("¿Coinciden?", np.allclose(verificacion, b))   # allclose: compara tolerando errores mínimos de redondeo

> ⚠️ **`@` vs `*`** es el error más común de la clase.
> - `A @ x` → **producto matricial** (el del álgebra).
> - `A * x` → multiplica elemento por elemento. No da error, **da otro número**. Peligrosísimo.

### Interpretación económica

Producir **90 sillas y 20 mesas** agota exactamente las 220 horas de carpintería y las 150 de pintura.
No sobra capacidad ociosa ni hace falta pagar horas extra.

Ese "exactamente" es lo que distingue un **sistema de ecuaciones** (buscamos igualdad) de la
**programación lineal** que vieron con Rita (donde hay *desigualdades*: usar *hasta* 220 horas, y
además maximizar la ganancia).

---
# ⚠️ Parte C — Cuándo el sistema no tiene solución

No todo sistema se puede resolver. El **determinante** de A avisa antes de intentarlo:

| Determinante | Qué significa | Interpretación en la fábrica |
|---|---|---|
| **≠ 0** | Solución única ✅ | Los sectores aportan información distinta |
| **= 0** | Ninguna solución, o infinitas ❌ | Una ecuación es "redundante": no aporta nada nuevo |

In [ ]:
print("Determinante de A:", np.linalg.det(A))   # det ≠ 0 → tiene solución única

In [ ]:
# Un caso patológico: la segunda máquina es exactamente el doble de la primera
A_mala = np.array([[2, 4],
                   [4, 8]])     # fila 2 = fila 1 × 2  → no aporta información nueva

print("Determinante:", np.linalg.det(A_mala))

try:
    np.linalg.solve(A_mala, np.array([220, 150]))
except np.linalg.LinAlgError as e:
    print("❌ Python no puede resolverlo:", e)

**Traducción al mundo real:** si el determinante da 0, el modelo está mal planteado. Alguien cargó
dos veces la misma restricción, o dos sectores son en realidad el mismo. **Es un error de datos, no de Python.**

---
# 📦 Parte D — Un caso más grande: planificación de producción

Una empresa de electrodomésticos fabrica tres productos y quiere agotar tres recursos escasos.

| Recurso | Producto A | Producto B | Producto C | Disponible |
|---|---|---|---|---|
| Horas de máquina | 3 | 2 | 4 | 2.400 |
| Horas de mano de obra | 2 | 5 | 3 | 2.300 |
| Kg de materia prima | 4 | 3 | 2 | 2.200 |

El planteo es idéntico, solo que ahora la matriz es 3×3.

In [ ]:
A3 = np.array([[3, 2, 4],     # horas de máquina por unidad de A, B y C
               [2, 5, 3],     # horas de mano de obra
               [4, 3, 2]])    # kg de materia prima

b3 = np.array([2400, 2300, 2200])

print("Determinante:", round(np.linalg.det(A3), 2))   # ≠ 0 → adelante

x3 = np.linalg.solve(A3, b3)

for producto, cantidad in zip(["A", "B", "C"], x3):
    print(f"Producto {producto}: {cantidad:,.1f} unidades")

In [ ]:
# Presentamos el resultado como una tabla, que es como se lo mandás al gerente
plan = pd.DataFrame({
    "Producto": ["A", "B", "C"],
    "Unidades": x3.round(1)
})
plan["Participación %"] = (plan["Unidades"] / plan["Unidades"].sum() * 100).round(1)

plan

> 💡 **Cuidado con las soluciones negativas.** Si alguna cantidad hubiera dado negativa, el sistema
> tendría solución *matemática* pero no *económica*: no se pueden producir −40 heladeras. Ahí es donde
> entra la programación lineal, que impone que las cantidades sean ≥ 0.

---
# 🧾 Parte E — Un proceso que se muerde la cola

Acá está el caso donde el sistema de ecuaciones deja de ser un ejercicio y se vuelve imprescindible.

**El contexto.** Un centro de servicios administra el proceso **P2P** (*Purchase to Pay*): desde que
llega la factura de un proveedor hasta que se le paga. El circuito tiene cuatro estaciones:

```
   ┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌──────┐
   │  Recepción  │──▶│ Validación  │──▶│ Aprobación  │──▶│ Pago │──▶ pagada
   └─────────────┘    └─────────────┘    └─────────────┘    └──────┘
          ▲                  ▲  │               ▲  │            │
          └──────────────────┘  └───────────────┘  └────────────┘
             18% rechazado         12% falta doc.     4% error de datos
```

**El problema.** Una factura rechazada **vuelve para atrás** y se procesa otra vez. Y al volver a
pasar, puede volver a fallar. El retrabajo se alimenta a sí mismo.

**La pregunta del gerente:** si entran 10.000 facturas por mes, ¿cuántas toca realmente cada estación?

> ⚠️ La respuesta **no es 10.000**. Y no se puede calcular sumando: hay que resolver un sistema,
> porque cada estación depende de estaciones que a su vez dependen de ella.

In [ ]:
flujos = pd.read_csv(URL + "proceso_p2p_flujos.csv")
estaciones = pd.read_csv(URL + "proceso_p2p_estaciones.csv")

flujos

Cada fila dice: *"de todo lo que procesa `origen`, esta proporción va a parar a `destino`"*.
Fijate que de Validación sale un 18% **hacia atrás** (a Recepción) y un 82% hacia adelante.

In [ ]:
etapas = ["Recepción", "Validación", "Aprobación", "Pago"]

# Armamos la matriz A. A[i, j] = fracción de lo que procesa la estación j que termina en la estación i
A = pd.DataFrame(0.0, index=etapas, columns=etapas)

for _, fila in flujos.iterrows():
    if fila["destino"] in etapas:              # "Completado" no es una estación, es la salida
        A.loc[fila["destino"], fila["origen"]] = fila["proporcion"]

A

Leé la matriz **por columnas**: la columna "Validación" dice que de todo lo que valida, un 0,18 vuelve
a Recepción y un 0,82 sigue a Aprobación.

El planteo es el mismo de siempre. Cada estación procesa lo que le entra de afuera **más** lo que le
mandan las otras:

$$x = \underbrace{d}_{\text{facturas nuevas}} + \underbrace{A\,x}_{\text{lo que le mandan las demás}}
\qquad \Longrightarrow \qquad (I - A)\,x = d$$

In [ ]:
d = estaciones["facturas_nuevas_mes"].values      # 10.000 entran por Recepción, 0 por el resto
I = np.eye(4)

print("Determinante de (I - A):", round(np.linalg.det(I - A.values), 5))   # ≠ 0 → se puede resolver

x = np.linalg.solve(I - A.values, d)

carga = pd.DataFrame({
    "estación": etapas,
    "facturas_nuevas": d,
    "carga_real": x.round(0),
    "veces_la_entrada": (x / 10000).round(2),
})
carga

### 😬 Ahí está el problema

Entran 10.000 facturas, pero **Validación procesa 13.927**: un 39% más. Y entre las cuatro estaciones
se producen casi **49.000 toques de factura** para pagar 10.000.

Ese 39% extra es trabajo que **no agrega ningún valor**: es la misma factura dando vueltas.

In [ ]:
# Control de consistencia: por Pago tienen que salir exactamente las 10.000 que entraron
salen = x[3] * 0.96      # el 96% de lo que pasa por Pago se completa

print(f"Facturas que entraron : {d.sum():>9,.0f}")
print(f"Facturas completadas  : {salen:>9,.0f}")
print(f"Total de toques       : {x.sum():>9,.0f}   ← {x.sum()/d.sum():.1f} toques por factura")

**Siempre hacé este control.** Si por la salida no sale lo mismo que entró, el modelo pierde o inventa
facturas y está mal planteado.

### ¿Para qué sirve el número? Para dimensionar el equipo

Cada estación tiene una productividad distinta. Con la carga real ya podemos calcular **cuánta gente
hace falta** — en la jerga, cuántos **FTE** (*Full Time Equivalent*, una persona a tiempo completo).

In [ ]:
carga["productividad"] = estaciones["productividad_mes_por_persona"].values
carga["FTE_necesarios"] = (carga["carga_real"] / carga["productividad"]).round(1)

# Con qué se compararía alguien que ignora el retrabajo
carga["FTE_si_ignoro_retrabajo"] = (10000 / carga["productividad"]).round(1)
carga["subestimación"] = (carga["FTE_necesarios"] - carga["FTE_si_ignoro_retrabajo"]).round(1)

carga[["estación", "carga_real", "FTE_necesarios", "FTE_si_ignoro_retrabajo", "subestimación"]]

In [ ]:
total_real = carga["FTE_necesarios"].sum()
total_ingenuo = carga["FTE_si_ignoro_retrabajo"].sum()

print(f"Equipo necesario (con retrabajo) : {total_real:>5.1f} FTE")
print(f"Equipo si ignoro el retrabajo    : {total_ingenuo:>5.1f} FTE")
print(f"{'-' * 45}")
print(f"Diferencia                       : {total_real - total_ingenuo:>5.1f} FTE de menos")

> 🎯 **Esto es exactamente lo que pasa en la vida real.** Alguien dimensiona el equipo con la regla de
> tres simple, contrata de menos, y después el área vive apagando incendios sin entender por qué.
> El retrabajo es invisible hasta que lo modelás.

### El business case: ¿cuánto vale arreglar el proceso?

Supongamos que se automatiza el matching de la factura contra la orden de compra, y el rechazo en
Validación baja del **18% al 8%**. ¿Cuánto se ahorra?

Volvemos a resolver el mismo sistema con el parámetro cambiado.

In [ ]:
def simular(rechazo_validacion):
    """Resuelve el sistema con otro % de rechazo y devuelve la carga y los FTE."""
    M = A.copy()
    M.loc["Recepción",  "Validación"] = rechazo_validacion        # lo que vuelve para atrás
    M.loc["Aprobación", "Validación"] = 1 - rechazo_validacion    # lo que sigue adelante
    carga_sim = np.linalg.solve(I - M.values, d)
    fte = (carga_sim / carga["productividad"].values).sum()
    return carga_sim, fte

carga_hoy, fte_hoy = simular(0.18)
carga_nueva, fte_nueva = simular(0.08)

comparacion = pd.DataFrame({
    "estación": etapas,
    "hoy (18%)": carga_hoy.round(0),
    "con mejora (8%)": carga_nueva.round(0),
    "diferencia": (carga_nueva - carga_hoy).round(0),
})
comparacion

In [ ]:
COSTO_FTE_ANUAL = 1_800_000     # costo anual de una persona, en pesos

ahorro_fte = fte_hoy - fte_nueva
ahorro_pesos = ahorro_fte * COSTO_FTE_ANUAL

print(f"Equipo hoy         : {fte_hoy:>5.1f} FTE")
print(f"Equipo con mejora  : {fte_nueva:>5.1f} FTE")
print(f"Ahorro             : {ahorro_fte:>5.1f} FTE")
print(f"\nAhorro anual: $ {ahorro_pesos:,.0f}")
print("\nSi el proyecto de automatización cuesta menos que eso, se paga solo en el primer año.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

pos = np.arange(len(etapas))
ancho = 0.35
ax.bar(pos - ancho/2, carga_hoy,   ancho, label="Hoy (18% de rechazo)",   color="#243b5e")
ax.bar(pos + ancho/2, carga_nueva, ancho, label="Con mejora (8%)",        color="#e07b39")
ax.axhline(10000, color="gray", linestyle="--", linewidth=1.5,
           label="Facturas que realmente entran (10.000)")

ax.set_xticks(pos)
ax.set_xticklabels(etapas)
ax.set_ylabel("Facturas procesadas por mes")
ax.set_title("Carga real de cada estación del proceso P2P", loc="left", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Todo lo que está **por encima de la línea gris es retrabajo**. El gráfico muestra de un vistazo dónde
duele y cuánto se recupera arreglándolo.

---
# 💰 Parte F — ¿Cuánto cuesta *realmente* procesar una factura?

Segundo caso, misma matemática. Ahora el problema es de **costos**.

El centro de servicios tiene seis áreas:

| Tipo | Áreas | Qué hacen |
|---|---|---|
| **Operativas** | P2P, OTC, R2R | Le dan servicio al negocio |
| **De soporte** | IT, RRHH, Facilities | Le dan servicio a las operativas... **y entre ellas** |

**El problema:** IT le da soporte a RRHH (sistemas de nómina), pero RRHH le da servicio a IT
(selección, capacitación, liquidación de sueldos). ¿Cuál se reparte primero? **Se muerden la cola,
igual que las facturas.**

In [ ]:
costos = pd.read_csv(URL + "gbs_costos.csv")
reparto = pd.read_csv(URL + "gbs_reparto.csv").set_index("presta")

costos

In [ ]:
reparto      # cada fila suma 1: es el 100% del servicio que presta esa área

El planteo, otra vez, es el mismo:

$$C_i = \underbrace{D_i}_{\text{costo propio}} + \underbrace{\sum_j a_{ij} C_j}_{\text{lo que le facturan las otras áreas de soporte}}$$

$$\Longrightarrow \quad (I - A)\,C = D$$

In [ ]:
soporte = ["IT", "RRHH", "Facilities"]
operativas = ["P2P", "OTC", "R2R"]

D_directo = costos.set_index("centro")["costo_directo_mensual"]

# A[i, j] = fracción del área j que consume el área i (solo entre áreas de soporte)
A_costos = reparto.loc[soporte, soporte].T.values

C = np.linalg.solve(np.eye(3) - A_costos, D_directo[soporte].values)

resultado = pd.DataFrame({
    "costo_directo": D_directo[soporte].values,
    "costo_total": C.round(0),
    "multiplicador": (C / D_directo[soporte].values).round(2),
}, index=soporte)

resultado

**RRHH tiene un multiplicador de 2,02**: su costo directo es de $2,4 millones, pero el costo *real* de
hacerlo funcionar es de $4,85 millones, porque consume muchísimo IT.

> 💡 Ese multiplicador es un argumento de negociación potente. Cuando un área de soporte dice
> "yo solo cuesto 2 millones", el número real puede ser más del doble.

In [ ]:
# Ahora repartimos el costo total de las áreas de soporte entre las operativas
costo_final = {}
for op in operativas:
    costo_final[op] = D_directo[op] + sum(C[i] * reparto.loc[s, op] for i, s in enumerate(soporte))

final = pd.DataFrame({
    "costo_directo": [D_directo[op] for op in operativas],
    "costo_con_soporte": [costo_final[op] for op in operativas],
    "volumen_mensual": [costos.set_index("centro").loc[op, "volumen_mensual"] for op in operativas],
}, index=operativas)

final["costo_x_transacción_directo"] = (final["costo_directo"] / final["volumen_mensual"]).round(0)
final["costo_x_transacción_real"]    = (final["costo_con_soporte"] / final["volumen_mensual"]).round(0)
final["subestimación_%"] = (100 * (final["costo_x_transacción_real"] / final["costo_x_transacción_directo"] - 1)).round(0)

final[["costo_x_transacción_directo", "costo_x_transacción_real", "subestimación_%"]]

In [ ]:
# Control: nada se pierde ni se crea. Todo el costo directo termina en las operativas.
print(f"Suma de costos directos    : $ {D_directo.sum():>12,.0f}")
print(f"Suma repartida a operativas: $ {sum(costo_final.values()):>12,.0f}")
print(f"¿Cierra? {np.isclose(D_directo.sum(), sum(costo_final.values()))}")

### El resultado que importa

Procesar una factura de P2P **no cuesta $198: cuesta $265**. Un 34% más.

Esa diferencia es la que decide cosas concretas:

- **Tercerizar o no.** Si un proveedor externo cobra $230 por factura, con el costo "directo" parece
  carísimo; con el costo real, es más barato que hacerlo adentro.
- **Cuánto cobrarle a cada unidad de negocio** por el servicio.
- **Dónde conviene automatizar.**

> 🎯 Decidir con el costo directo es el error de costeo más caro y más común que existe. Y evitarlo
> costó exactamente una línea: `np.linalg.solve`.

> 📎 **Nota al margen.** Este mismo modelo, aplicado a los sectores de un país entero en lugar de las
> áreas de una empresa, se llama **matriz insumo-producto de Leontief** y le valió el Nobel de Economía
> en 1973. La matemática es idéntica: cambia la escala.

---
# 📝 Ejercicios

**Ejercicio 1.** Una panadería produce pan y facturas. Cada kilo de pan usa 0,5 kg de harina y 10 minutos
de horno; cada kilo de facturas usa 0,4 kg de harina y 20 minutos de horno. Hay 100 kg de harina y
3.000 minutos de horno. Planteá el sistema y resolvelo.

In [ ]:
# Tu respuesta acá

**Ejercicio 2.** Verificá tu resultado del ejercicio 1 con `@` y `np.allclose`. ¿Sobra algún recurso?

In [ ]:
# Tu respuesta acá

**Ejercicio 3.** En el proceso P2P, ¿qué pasa si el rechazo de Aprobación sube del 12% al 25%
(por ejemplo, porque cambió la política de aprobaciones)? Usá la matriz `A`, modificá esa celda
y volvé a resolver. ¿Qué estación sufre más?

In [ ]:
# Tu respuesta acá

**Ejercicio 4.** El gerente puede invertir en **una sola** mejora: bajar el rechazo de Validación del
18% al 10%, **o** el de Aprobación del 12% al 4%. Calculá el ahorro en FTE de cada una y recomendá.

In [ ]:
# Tu respuesta acá

**Ejercicio 5.** En el caso de costos, ¿cuál sería el costo por transacción de OTC si IT le dedicara
el 25% de su capacidad en vez del 40% (y esa diferencia fuera a P2P)? Modificá `reparto` y recalculá.

In [ ]:
# Tu respuesta acá

**Ejercicio 6 (integrador 🥇⚡🤓).** Un proveedor externo ofrece procesar todo el P2P a **$240 por
factura**. Con los dos análisis de la clase, armá el argumento a favor y en contra de aceptar.
Pista: ¿qué pasa con el costo de las áreas de soporte si P2P deja de existir?

In [ ]:
# Tu respuesta acá

---
## 🧭 Para llevarse

| Concepto | Comando |
|---|---|
| Resolver $Ax = b$ | `np.linalg.solve(A, b)` |
| Producto matricial | `A @ x` (¡no `A * x`!) |
| Verificar la solución | `np.allclose(A @ x, b)` |
| ¿Tiene solución? | `np.linalg.det(A) != 0` |
| Matriz identidad | `np.eye(n)` |
| Procesos que se retroalimentan | `np.linalg.solve(I - A, d)` |

**Las tres ideas de la clase:**

1. Cuando en un proceso **algo vuelve para atrás** —un retrabajo, un servicio cruzado, una devolución—
   no se puede calcular sumando. Hay que resolver un sistema.
2. La estructura $(I - A)\,x = d$ aparece en todos lados: carga de trabajo, costeo, cadenas de
   suministro, cobranzas. **Reconocerla es la mitad del trabajo.**
3. Siempre hacé el **control de consistencia**: que salga lo mismo que entró, que el costo total se
   conserve. Si no cierra, el modelo está mal y los números son basura convincente.